# DEAI – Datawarehouse uitbreiding: Retouren & Cursussen (Great Outdoors)

Deze notebook werkt **twee nieuwe knelpunten** uit binnen het bestaande Great Outdoors DWH:

* **Knelpunt 2 – Retourstromen**: welke producten worden vaak teruggestuurd, in welke regio's,
  om welke redenen en wat kost dat?
* **Knelpunt 4 – Cursussen**: wie volgt welke cursus, wat zijn de (aangenomen) kosten en hangt
  cursusdeelname samen met de tevredenheid van medewerkers?

### Hoe gebruik je deze notebook (1 knop)
1. **Draai eerst** de hoofd-notebook `GO_DWH_Bestelgedrag_Notebook.ipynb`. Die bouwt `GO_DWH.db`
   met de basis-dimensies (`dim_date`, `dim_product`, `dim_customer`, `dim_region`,
   `dim_sales_staff`) en `fact_order_sales`. Die hergebruiken we hier.
2. Open daarna deze notebook en kies **Run All** (alles uitvoeren). De laatste codecel roept
   `run_extra_pipeline()` aan: dat doet alles in één keer.

### Wat deze notebook toevoegt aan `GO_DWH.db`
* Nieuwe dimensies: `dim_return_reason` (SCD-type 2), `dim_course` (SCD-type 2),
  `dim_satisfaction_type` (SCD-type 1).
* Nieuwe feiten: `fact_returns`, `fact_training`, `fact_staff_satisfaction`.
* Analyse-views (voor controle en eenvoudige rapportage) en CSV-exports voor Power BI.

> **Slimme truc (retouren):** `fact_order_sales` bevat per `order_detail_bk` al de opgezochte
> sleutels (product, klant, regio, medewerker) én de verkoopprijs. We koppelen de retouren daar
> simpelweg aan vast, zodat we die opzoekwerk niet hoeven over te doen.

## 1. Imports en paden

In [1]:
from pathlib import Path
import sqlite3
import logging
from datetime import date
import numpy as np
import pandas as pd

# Deze notebook gaat ervan uit dat hij in dezelfde map staat als de databases.
BASE_DIR = Path.cwd()

SDM_DB = BASE_DIR / "GO_SDM.db"     # bron: hier staat alle ruwe (gestage) data
DWH_DB = BASE_DIR / "GO_DWH.db"     # doel: hier voegen we onze nieuwe tabellen aan toe

EXPORT_DIR = BASE_DIR / "powerbi_exports"   # CSV's voor Power BI
EXPORT_DIR.mkdir(exist_ok=True)

LOG_DIR = BASE_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)
EXTRA_LOG_PATH = LOG_DIR / "go_dwh_extra_etl.log"

print("SDM-bron :", SDM_DB)
print("DWH-doel :", DWH_DB)
print("Export   :", EXPORT_DIR)
print("Logbestand:", EXTRA_LOG_PATH)

SDM-bron : C:\Users\skyde\Documents\Github\DEAI-SE4\Great_Outdoors\GO_SDM.db
DWH-doel : C:\Users\skyde\Documents\Github\DEAI-SE4\Great_Outdoors\GO_DWH.db
Export   : C:\Users\skyde\Documents\Github\DEAI-SE4\Great_Outdoors\powerbi_exports
Logbestand: C:\Users\skyde\Documents\Github\DEAI-SE4\Great_Outdoors\logs\go_dwh_extra_etl.log


## 2. Logging instellen
Elke stap schrijft een regel naar `logs/go_dwh_extra_etl.log`. Zo kun je in Power BI later de
**pijplijnkwaliteit** laten zien (hoeveel rijen geladen, wanneer, met welke melding).

In [2]:
logger = logging.getLogger("go_dwh_extra_etl")
logger.setLevel(logging.INFO)

# Oude handlers weghalen zodat we niet dubbel loggen als de cel opnieuw draait.
if logger.hasHandlers():
    logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s|%(levelname)s|%(message)s", datefmt="%Y-%m-%d %H:%M:%S")
file_handler = logging.FileHandler(EXTRA_LOG_PATH, encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.propagate = False

def log_event(level, process_step, table_name="", action="", row_count="", details=""):
    # Eén vaste indeling: stap|tabel|actie|aantal_rijen|details
    message = f"{process_step}|{table_name}|{action}|{row_count}|{details}"
    if level == "ERROR":
        logger.error(message)
    elif level == "WARNING":
        logger.warning(message)
    else:
        logger.info(message)

log_event("INFO", "INIT", "logging", "READY", "", "Logging gestart")
print("Logging staat aan.")

Logging staat aan.


## 3. Hulpfuncties (verbinden en lezen)

In [3]:
def connect(db_path):
    # Verbinding met foreign keys AAN, zodat verkeerde sleutels worden tegengehouden.
    con = sqlite3.connect(db_path)
    con.execute("PRAGMA foreign_keys = ON")
    return con

def read_sdm(table):
    # Leest een tabel uit het SDM (de bron).
    con = sqlite3.connect(SDM_DB)
    df = pd.read_sql_query(f'SELECT * FROM "{table}"', con)
    con.close()
    log_event("INFO", "EXTRACT_SDM", table, "READ", len(df), "Gelezen uit SDM")
    return df

def read_dwh(sql):
    # Voert een SELECT uit op het DWH en geeft het resultaat als tabel terug.
    con = connect(DWH_DB)
    df = pd.read_sql_query(sql, con)
    con.close()
    return df

def to_py(value):
    # numpy-getallen omzetten naar gewone Python-getallen (anders klaagt sqlite).
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    return value

## 4. Controle: is het basis-DWH klaar?
Deze notebook bouwt vóórt op de hoofd-notebook. We stoppen met een duidelijke melding als
`GO_DWH.db` of de basis-dimensies nog niet bestaan.

In [4]:
def check_dwh_ready():
    if not DWH_DB.exists():
        raise FileNotFoundError(
            "GO_DWH.db bestaat nog niet. Draai eerst GO_DWH_Bestelgedrag_Notebook.ipynb!"
        )
    bestaande = read_dwh("SELECT name FROM sqlite_master WHERE type='table'")["name"].tolist()
    nodig = ["dim_date", "dim_product", "dim_customer", "dim_region",
             "dim_sales_staff", "fact_order_sales"]
    ontbreekt = [t for t in nodig if t not in bestaande]
    if ontbreekt:
        raise RuntimeError(
            "Deze basis-tabellen ontbreken in het DWH (draai eerst de hoofd-notebook): "
            + ", ".join(ontbreekt)
        )
    log_event("INFO", "CHECK", "GO_DWH", "OK", "", "Basis-dimensies aanwezig")
    print("Basis-DWH is aanwezig. We kunnen verder.")

check_dwh_ready()

Basis-DWH is aanwezig. We kunnen verder.


## 5. Schema van de nieuwe tabellen
Hieronder de `CREATE TABLE`s. De SCD-type-2 dimensies (`dim_return_reason`, `dim_course`) hebben
extra kolommen: `valid_from`, `valid_to`, `is_current`, `version` om de historie bij te houden.

In [5]:
EXTRA_SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

-- ====================== Knelpunt 2: Retouren ======================
CREATE TABLE IF NOT EXISTS dim_return_reason (
    return_reason_key  INTEGER PRIMARY KEY AUTOINCREMENT,
    return_reason_bk   INTEGER NOT NULL,
    reason_description TEXT,
    reason_category    TEXT,        -- AFGELEID: Kwaliteit / Logistiek / Klantkeuze
    valid_from         TEXT,        -- SCD2
    valid_to           TEXT,        -- SCD2
    is_current         INTEGER,     -- SCD2 (1 = huidige versie)
    version            INTEGER      -- SCD2
);

CREATE TABLE IF NOT EXISTS fact_returns (
    return_key        INTEGER PRIMARY KEY AUTOINCREMENT,
    return_bk         INTEGER NOT NULL UNIQUE,
    date_key          INTEGER,
    product_key       INTEGER,
    customer_key      INTEGER,
    region_key        INTEGER,
    sales_staff_key   INTEGER,
    return_reason_key INTEGER,
    return_quantity   INTEGER,      -- aantal geretourneerd
    original_quantity INTEGER,      -- oorspronkelijk besteld aantal (voor retourpercentage)
    unit_sale_price   REAL,
    return_value      REAL,         -- AFGELEID: gederfde omzet = aantal * verkoopprijs
    gross_loss        REAL,         -- AFGELEID: gederfde marge = aantal * (verkoopprijs - kostprijs)
    FOREIGN KEY (date_key)          REFERENCES dim_date(date_key),
    FOREIGN KEY (product_key)       REFERENCES dim_product(product_key),
    FOREIGN KEY (customer_key)      REFERENCES dim_customer(customer_key),
    FOREIGN KEY (region_key)        REFERENCES dim_region(region_key),
    FOREIGN KEY (sales_staff_key)   REFERENCES dim_sales_staff(sales_staff_key),
    FOREIGN KEY (return_reason_key) REFERENCES dim_return_reason(return_reason_key)
);

-- ====================== Knelpunt 4: Cursussen ======================
CREATE TABLE IF NOT EXISTS dim_course (
    course_key         INTEGER PRIMARY KEY AUTOINCREMENT,
    course_bk          INTEGER NOT NULL,
    course_description TEXT,
    course_category    TEXT,        -- AFGELEID: Orientatie/Communicatie/Sales/Marketing/Management
    standard_cost      REAL,        -- AFGELEID (aanname): kost per cursus, niet in bron aanwezig
    valid_from         TEXT,        -- SCD2
    valid_to           TEXT,        -- SCD2
    is_current         INTEGER,     -- SCD2
    version            INTEGER      -- SCD2
);

CREATE TABLE IF NOT EXISTS dim_satisfaction_type (
    satisfaction_type_key INTEGER PRIMARY KEY AUTOINCREMENT,
    satisfaction_type_bk  INTEGER NOT NULL UNIQUE,
    type_description      TEXT,
    satisfaction_score    INTEGER  -- AFGELEID: numerieke score 1..5
);

CREATE TABLE IF NOT EXISTS fact_training (
    training_key            INTEGER PRIMARY KEY AUTOINCREMENT,
    year_num                INTEGER,
    sales_representative_bk INTEGER,
    sales_staff_key         INTEGER,
    course_key              INTEGER,
    courses_followed        INTEGER,  -- altijd 1 (één regel = één gevolgde cursus)
    course_cost             REAL,     -- AFGELEID: kost van die cursus (uit dim_course)
    FOREIGN KEY (sales_staff_key) REFERENCES dim_sales_staff(sales_staff_key),
    FOREIGN KEY (course_key)      REFERENCES dim_course(course_key)
);

CREATE TABLE IF NOT EXISTS fact_staff_satisfaction (
    satisfaction_key        INTEGER PRIMARY KEY AUTOINCREMENT,
    year_num                INTEGER,
    sales_representative_bk INTEGER,
    sales_staff_key         INTEGER,
    satisfaction_type_key   INTEGER,
    satisfaction_score      INTEGER,
    FOREIGN KEY (sales_staff_key)       REFERENCES dim_sales_staff(sales_staff_key),
    FOREIGN KEY (satisfaction_type_key) REFERENCES dim_satisfaction_type(satisfaction_type_key)
);
"""

def create_extra_schema():
    con = connect(DWH_DB)
    con.executescript(EXTRA_SCHEMA_SQL)
    con.commit()
    con.close()
    log_event("INFO", "SCHEMA", "extra", "CREATE", 1, "Nieuwe tabellen aangemaakt indien nodig")
    print("Nieuwe tabellen staan klaar.")

## 6. Feiten leegmaken (full reload)
Onze **inlaadstrategie** voor de feiten is *full reload*: we maken de feittabellen elke run leeg en
vullen ze opnieuw. Dat is eenvoudig en altijd consistent. De SCD-dimensies blijven juist staan,
zodat hun historie bewaard blijft.

In [6]:
def reset_extra_facts():
    con = connect(DWH_DB)
    for t in ["fact_returns", "fact_training", "fact_staff_satisfaction"]:
        con.execute(f"DELETE FROM {t}")
    # Tellers van AUTOINCREMENT terugzetten (netjes, mag falen als nog niet gebruikt).
    try:
        con.execute("DELETE FROM sqlite_sequence WHERE name IN "
                    "('fact_returns','fact_training','fact_staff_satisfaction')")
    except sqlite3.OperationalError:
        pass
    con.commit()
    con.close()
    log_event("INFO", "RESET", "facts", "DELETE", 1, "Feittabellen geleegd voor full reload")

## 7. SCD-functies (Slowly Changing Dimensions)
* **SCD-type 1** (`dim_satisfaction_type`): we overschrijven gewoon de oude waarden. Geen historie.
* **SCD-type 2** (`dim_return_reason`, `dim_course`): als een waarde wijzigt, sluiten we de oude rij
  af (`is_current = 0`, `valid_to = vandaag`) en voegen we een **nieuwe** rij toe (`version + 1`).
  Zo blijft de geschiedenis bewaard. Bij de allereerste run worden alle rijen als versie 1 ingeladen.

In [7]:
def load_scd1(con, table, df):
    # Type 1: tabel leegmaken en opnieuw vullen (overschrijven).
    con.execute(f"DELETE FROM {table}")
    df.to_sql(table, con, if_exists="append", index=False)
    con.commit()
    log_event("INFO", "LOAD_SCD1", table, "REPLACE", len(df), "Type-1 dimensie herladen")


def _insert_scd2(con, table, bk_col, attr_cols, row, today, version):
    cols = [bk_col] + attr_cols + ["valid_from", "valid_to", "is_current", "version"]
    vals = [to_py(row[bk_col])] + [to_py(row[a]) for a in attr_cols] + [today, None, 1, version]
    placeholders = ",".join(["?"] * len(cols))
    con.execute(f"INSERT INTO {table} ({','.join(cols)}) VALUES ({placeholders})", vals)


def load_scd2(con, table, key_col, bk_col, attr_cols, df):
    today = date.today().isoformat()
    huidig = pd.read_sql_query(f"SELECT * FROM {table} WHERE is_current = 1", con)

    nieuw = 0
    gewijzigd = 0
    for _, row in df.iterrows():
        bk = to_py(row[bk_col])
        match = huidig[huidig[bk_col] == bk] if not huidig.empty else huidig

        if match.empty:
            # Onbekende business key -> nieuwe rij, versie 1.
            _insert_scd2(con, table, bk_col, attr_cols, row, today, version=1)
            nieuw += 1
        else:
            cur = match.iloc[0]
            is_anders = any(str(cur[a]) != str(row[a]) for a in attr_cols)
            if is_anders:
                # Oude versie afsluiten ...
                con.execute(
                    f"UPDATE {table} SET is_current = 0, valid_to = ? WHERE {key_col} = ?",
                    (today, int(cur[key_col]))
                )
                # ... en nieuwe versie toevoegen.
                _insert_scd2(con, table, bk_col, attr_cols, row, today, version=int(cur["version"]) + 1)
                gewijzigd += 1

    con.commit()
    log_event("INFO", "LOAD_SCD2", table, "UPSERT", nieuw + gewijzigd,
              f"nieuw={nieuw}, gewijzigd={gewijzigd}")
    print(f"{table}: {nieuw} nieuw, {gewijzigd} gewijzigd (SCD2).")

## 8. Dimensie `dim_return_reason` laden (SCD2 + afgeleide categorie)

In [8]:
def load_dim_return_reason():
    df = read_sdm("sales_return_reason").copy()
    df["return_reason_bk"] = df["RETURN_REASON_CODE"].astype(int)
    df["reason_description"] = df["RETURN_DESCRIPTION_EN"].astype(str)

    # AFGELEIDE categorie: groepeer de redenen in 3 begrijpelijke groepen.
    categorie = {1: "Kwaliteit", 2: "Logistiek", 3: "Klantkeuze", 4: "Logistiek", 5: "Kwaliteit"}
    df["reason_category"] = df["return_reason_bk"].map(categorie).fillna("Overig")

    df = df[["return_reason_bk", "reason_description", "reason_category"]]
    con = connect(DWH_DB)
    load_scd2(con, "dim_return_reason", "return_reason_key", "return_reason_bk",
              ["reason_description", "reason_category"], df)
    con.close()

## 9. Dimensie `dim_course` laden (SCD2 + afgeleide categorie en kosten)

In [9]:
def bepaal_categorie(omschrijving):
    d = omschrijving.lower()
    if "orientation" in d:    return "Orientatie"
    if "communication" in d:  return "Communicatie"
    if "sales" in d:          return "Sales"
    if "marketing" in d:      return "Marketing"
    if "management" in d:     return "Management"
    return "Overig"

def load_dim_course():
    df = read_sdm("staff_course").copy()
    df["course_bk"] = df["COURSE_CODE"].astype(int)
    df["course_description"] = df["COURSE_DESCRIPTION"].astype(str)
    df["course_category"] = df["course_description"].map(bepaal_categorie)

    # AFGELEIDE (aangenomen) kosten per cursuscategorie. De bron bevat GEEN kosten;
    # dit is een gedocumenteerde aanname zodat de KPI 'kosten per cursus' getoond kan worden.
    kosten = {"Orientatie": 250.0, "Communicatie": 400.0, "Sales": 600.0,
              "Marketing": 750.0, "Management": 1000.0, "Overig": 500.0}
    df["standard_cost"] = df["course_category"].map(kosten)

    df = df[["course_bk", "course_description", "course_category", "standard_cost"]]
    con = connect(DWH_DB)
    load_scd2(con, "dim_course", "course_key", "course_bk",
              ["course_description", "course_category", "standard_cost"], df)
    con.close()

## 10. Dimensie `dim_satisfaction_type` laden (SCD1 + afgeleide score)

In [10]:
def load_dim_satisfaction_type():
    df = read_sdm("staff_satisfaction_type").copy()
    df["satisfaction_type_bk"] = df["SATISFACTION_TYPE_CODE"].astype(int)
    df["type_description"] = df["SATISFACTION_TYPE_DESCRIPTION"].astype(str)
    # AFGELEIDE numerieke score: code 1..5 loopt van 'Not satisfied' tot 'More than satisfied'.
    df["satisfaction_score"] = df["satisfaction_type_bk"]
    df = df[["satisfaction_type_bk", "type_description", "satisfaction_score"]]
    con = connect(DWH_DB)
    load_scd1(con, "dim_satisfaction_type", df)
    con.close()

## 11. Feit `fact_returns` laden
We koppelen elke retourregel aan `fact_order_sales` (op `order_detail_bk`). Daaruit komen kant-en-klaar
de sleutels (product, klant, regio, medewerker) en de verkoop-/kostprijs. Retourdatums die nog niet in
`dim_date` staan, voegen we eerst toe.

In [11]:
def add_missing_dates(dt_series):
    # Voegt datums toe aan dim_date die er nog niet in staan (INSERT OR IGNORE).
    datums = pd.to_datetime(dt_series.dropna().dt.date.astype(str)).drop_duplicates()
    con = connect(DWH_DB)
    cur = con.cursor()
    toegevoegd = 0
    for d in datums:
        cur.execute(
            """INSERT OR IGNORE INTO dim_date
               (date_key, full_date, day_of_month, month_num, month_name, quarter_num,
                year_num, week_num, day_name, is_weekend)
               VALUES (?,?,?,?,?,?,?,?,?,?)""",
            (int(d.strftime("%Y%m%d")), d.strftime("%Y-%m-%d"), int(d.day), int(d.month),
             d.strftime("%B"), (int(d.month) - 1) // 3 + 1, int(d.year),
             int(d.isocalendar().week), d.strftime("%A"), 1 if d.weekday() >= 5 else 0)
        )
        toegevoegd += cur.rowcount
    con.commit()
    con.close()
    log_event("INFO", "DIM_DATE", "dim_date", "INSERT_MISSING", toegevoegd, "Retourdatums toegevoegd")
    print(f"dim_date: {toegevoegd} ontbrekende retourdatums toegevoegd.")


def load_fact_returns():
    returns = read_sdm("sales_returned_item").copy()

    # Datum omzetten naar date_key (YYYYMMDD), zelfde formaat als de hoofd-notebook.
    dt = pd.to_datetime(returns["RETURN_DATE"], format="%d-%b-%Y %I:%M:%S %p", errors="coerce")
    add_missing_dates(dt)
    returns["date_key"] = dt.dt.strftime("%Y%m%d").astype("Int64")

    returns["return_bk"] = returns["RETURN_CODE"].astype(int)
    returns["order_detail_bk"] = pd.to_numeric(returns["ORDER_DETAIL_CODE"], errors="coerce").astype("Int64")
    returns["return_quantity"] = pd.to_numeric(returns["RETURN_QUANTITY"], errors="coerce").fillna(0).astype(int)
    returns["return_reason_bk"] = returns["RETURN_REASON_CODE"].astype(int)

    # Hergebruik fact_order_sales: hier zitten alle sleutels + prijzen al in.
    fos = read_dwh("""
        SELECT order_detail_bk, product_key, customer_key, region_key, sales_staff_key,
               unit_sale_price, unit_cost, quantity AS original_quantity
        FROM fact_order_sales
    """)
    fos["order_detail_bk"] = pd.to_numeric(fos["order_detail_bk"], errors="coerce").astype("Int64")

    m = returns.merge(fos, on="order_detail_bk", how="left")

    # Koppel de retourreden aan de HUIDIGE versie in dim_return_reason.
    rr = read_dwh("SELECT return_reason_key, return_reason_bk FROM dim_return_reason WHERE is_current = 1")
    reason_map = rr.set_index("return_reason_bk")["return_reason_key"].to_dict()
    m["return_reason_key"] = m["return_reason_bk"].map(reason_map)

    # Afgeleide meetwaarden.
    m["unit_sale_price"] = m["unit_sale_price"].fillna(0)
    m["return_value"] = m["return_quantity"] * m["unit_sale_price"]
    m["gross_loss"] = m["return_quantity"] * (m["unit_sale_price"] - m["unit_cost"].fillna(0))

    out = m[["return_bk", "date_key", "product_key", "customer_key", "region_key",
             "sales_staff_key", "return_reason_key", "return_quantity", "original_quantity",
             "unit_sale_price", "return_value", "gross_loss"]]

    ongekoppeld = int(out["product_key"].isna().sum())
    if ongekoppeld:
        log_event("WARNING", "LOAD_FACT", "fact_returns", "NO_ORDER_MATCH", ongekoppeld,
                  "Retouren zonder gekoppelde orderregel")

    con = connect(DWH_DB)
    out.to_sql("fact_returns", con, if_exists="append", index=False)
    con.commit()
    con.close()
    log_event("INFO", "LOAD_FACT", "fact_returns", "INSERT", len(out), "Retouren geladen")
    print(f"fact_returns: {len(out)} rijen geladen.")

## 12. Feit `fact_training` laden
Elke regel = één medewerker die in een bepaald jaar één cursus volgde. De medewerker (`SALES_REPRESENTATIVE_CODE`)
koppelen we aan `dim_sales_staff` (`sales_staff_bk`) — dit zijn dezelfde personen, ook al heten de tabellen
in de bron-databases anders (de *database-overschrijdende associatie* uit de casus).

In [12]:
def load_fact_training():
    tr = read_sdm("staff_training").copy()
    tr["year_num"] = tr["YEAR"].astype(int)
    tr["sales_representative_bk"] = tr["SALES_REPRESENTATIVE_CODE"].astype(int)
    tr["course_bk"] = tr["COURSE_CODE"].astype(int)

    # Medewerker-sleutel: rep-code == sales_staff_code.
    ss = read_dwh("SELECT sales_staff_key, sales_staff_bk FROM dim_sales_staff")
    ss_map = ss.set_index("sales_staff_bk")["sales_staff_key"].to_dict()
    tr["sales_staff_key"] = tr["sales_representative_bk"].map(ss_map)

    # Cursus-sleutel + kosten uit de HUIDIGE versie van dim_course.
    cc = read_dwh("SELECT course_key, course_bk, standard_cost FROM dim_course WHERE is_current = 1")
    tr["course_key"] = tr["course_bk"].map(cc.set_index("course_bk")["course_key"].to_dict())
    tr["course_cost"] = tr["course_bk"].map(cc.set_index("course_bk")["standard_cost"].to_dict())
    tr["courses_followed"] = 1

    onbekend = int(tr["sales_staff_key"].isna().sum())
    if onbekend:
        log_event("WARNING", "LOAD_FACT", "fact_training", "UNKNOWN_STAFF", onbekend,
                  "Trainingen van medewerkers zonder verkooprol (sales_staff_key = NULL)")

    out = tr[["year_num", "sales_representative_bk", "sales_staff_key",
              "course_key", "courses_followed", "course_cost"]]
    con = connect(DWH_DB)
    out.to_sql("fact_training", con, if_exists="append", index=False)
    con.commit()
    con.close()
    log_event("INFO", "LOAD_FACT", "fact_training", "INSERT", len(out), "Trainingen geladen")
    print(f"fact_training: {len(out)} rijen geladen ({onbekend} zonder verkooprol).")

## 13. Feit `fact_staff_satisfaction` laden

In [13]:
def load_fact_staff_satisfaction():
    sat = read_sdm("staff_satisfaction").copy()
    sat["year_num"] = sat["YEAR"].astype(int)
    sat["sales_representative_bk"] = sat["SALES_REPRESENTATIVE_CODE"].astype(int)
    sat["satisfaction_type_bk"] = sat["SATISFACTION_TYPE_CODE"].astype(int)

    ss = read_dwh("SELECT sales_staff_key, sales_staff_bk FROM dim_sales_staff")
    sat["sales_staff_key"] = sat["sales_representative_bk"].map(
        ss.set_index("sales_staff_bk")["sales_staff_key"].to_dict())

    st = read_dwh("SELECT satisfaction_type_key, satisfaction_type_bk, satisfaction_score "
                  "FROM dim_satisfaction_type")
    sat["satisfaction_type_key"] = sat["satisfaction_type_bk"].map(
        st.set_index("satisfaction_type_bk")["satisfaction_type_key"].to_dict())
    sat["satisfaction_score"] = sat["satisfaction_type_bk"].map(
        st.set_index("satisfaction_type_bk")["satisfaction_score"].to_dict())

    out = sat[["year_num", "sales_representative_bk", "sales_staff_key",
               "satisfaction_type_key", "satisfaction_score"]]
    con = connect(DWH_DB)
    out.to_sql("fact_staff_satisfaction", con, if_exists="append", index=False)
    con.commit()
    con.close()
    log_event("INFO", "LOAD_FACT", "fact_staff_satisfaction", "INSERT", len(out), "Tevredenheid geladen")
    print(f"fact_staff_satisfaction: {len(out)} rijen geladen.")

## 14. Analyse-views
Handige views voor controle en voor wie liever direct SQL gebruikt. Power BI kan ze ook gewoon inladen.

In [14]:
ANALYSIS_VIEWS_SQL = """
DROP VIEW IF EXISTS vw_retouren_per_product;
CREATE VIEW vw_retouren_per_product AS
SELECT p.product_name, p.product_type, p.product_line,
       SUM(f.return_quantity) AS retour_aantal,
       SUM(f.return_value)    AS retour_waarde,
       SUM(f.gross_loss)      AS gederfde_marge,
       COUNT(*)               AS aantal_retouren
FROM fact_returns f
JOIN dim_product p ON f.product_key = p.product_key
GROUP BY p.product_name, p.product_type, p.product_line;

DROP VIEW IF EXISTS vw_retouren_per_reden;
CREATE VIEW vw_retouren_per_reden AS
SELECT r.reason_description, r.reason_category,
       SUM(f.return_quantity) AS retour_aantal,
       SUM(f.return_value)    AS retour_waarde,
       COUNT(*)               AS aantal_retouren
FROM fact_returns f
JOIN dim_return_reason r ON f.return_reason_key = r.return_reason_key
GROUP BY r.reason_description, r.reason_category;

DROP VIEW IF EXISTS vw_retouren_per_regio;
CREATE VIEW vw_retouren_per_regio AS
SELECT g.country, g.territory_name, g.region, g.city,
       SUM(f.return_quantity) AS retour_aantal,
       SUM(f.return_value)    AS retour_waarde
FROM fact_returns f
JOIN dim_region g ON f.region_key = g.region_key
GROUP BY g.country, g.territory_name, g.region, g.city;

DROP VIEW IF EXISTS vw_retourpercentage_per_product;
CREATE VIEW vw_retourpercentage_per_product AS
SELECT p.product_name, p.product_type, p.product_line,
       v.verkocht_aantal,
       COALESCE(r.retour_aantal, 0) AS retour_aantal,
       ROUND(100.0 * COALESCE(r.retour_aantal, 0) / v.verkocht_aantal, 2) AS retour_percentage
FROM (SELECT product_key, SUM(quantity) AS verkocht_aantal
      FROM fact_order_sales GROUP BY product_key) v
JOIN dim_product p ON v.product_key = p.product_key
LEFT JOIN (SELECT product_key, SUM(return_quantity) AS retour_aantal
           FROM fact_returns GROUP BY product_key) r ON v.product_key = r.product_key;

DROP VIEW IF EXISTS vw_cursus_deelname;
CREATE VIEW vw_cursus_deelname AS
SELECT c.course_description, c.course_category, c.standard_cost,
       COUNT(*)                                AS aantal_deelnames,
       COUNT(DISTINCT t.sales_representative_bk) AS aantal_medewerkers,
       SUM(t.course_cost)                       AS totale_kosten
FROM fact_training t
JOIN dim_course c ON t.course_key = c.course_key AND c.is_current = 1
GROUP BY c.course_description, c.course_category, c.standard_cost;

DROP VIEW IF EXISTS vw_tevredenheid_per_jaar;
CREATE VIEW vw_tevredenheid_per_jaar AS
SELECT year_num,
       ROUND(AVG(satisfaction_score), 2) AS gem_tevredenheid,
       COUNT(*)                          AS aantal_metingen
FROM fact_staff_satisfaction
GROUP BY year_num;

DROP VIEW IF EXISTS vw_training_vs_tevredenheid;
CREATE VIEW vw_training_vs_tevredenheid AS
SELECT s.sales_representative_bk,
       COALESCE(t.aantal_cursussen, 0)     AS aantal_cursussen,
       ROUND(AVG(s.satisfaction_score), 2) AS gem_tevredenheid
FROM fact_staff_satisfaction s
LEFT JOIN (SELECT sales_representative_bk, COUNT(*) AS aantal_cursussen
           FROM fact_training GROUP BY sales_representative_bk) t
       ON s.sales_representative_bk = t.sales_representative_bk
GROUP BY s.sales_representative_bk, t.aantal_cursussen;
"""

def create_analysis_views():
    con = connect(DWH_DB)
    con.executescript(ANALYSIS_VIEWS_SQL)
    con.commit()
    con.close()
    log_event("INFO", "VIEWS", "extra", "CREATE", 1, "Analyse-views aangemaakt")
    print("Analyse-views aangemaakt.")

## 15. Export naar CSV (voor Power BI)
Power BI leest SQLite niet makkelijk. Daarom exporteren we de nieuwe tabellen (plus de gedeelde
dimensies) naar CSV in de map `powerbi_exports/`. Die importeer je in Power BI.

In [15]:
def export_csv():
    tabellen = [
        "dim_return_reason", "fact_returns",
        "dim_course", "dim_satisfaction_type", "fact_training", "fact_staff_satisfaction",
        "dim_date", "dim_product", "dim_region", "dim_customer", "dim_sales_staff",
    ]
    con = connect(DWH_DB)
    for t in tabellen:
        df = pd.read_sql_query(f"SELECT * FROM {t}", con)
        df.to_csv(EXPORT_DIR / f"{t}.csv", index=False, encoding="utf-8-sig")
        log_event("INFO", "EXPORT_CSV", t, "WRITE", len(df), "CSV geschreven")
    con.close()
    print("CSV-bestanden geschreven naar:", EXPORT_DIR)

## 16. De volledige pijplijn (één knop)
`run_extra_pipeline()` voert alle stappen in de juiste volgorde uit en geeft een rijtelling terug.

In [16]:
def run_extra_pipeline():
    log_event("INFO", "RUN", "", "START", "", "Start extra DWH ETL (retouren + cursussen)")

    check_dwh_ready()
    create_extra_schema()
    reset_extra_facts()

    # 1) dimensies
    load_dim_return_reason()
    load_dim_course()
    load_dim_satisfaction_type()

    # 2) feiten
    load_fact_returns()
    load_fact_training()
    load_fact_staff_satisfaction()

    # 3) views + export
    create_analysis_views()
    export_csv()

    # 4) controle: rijtelling
    con = connect(DWH_DB)
    overzicht = []
    for t in ["dim_return_reason", "dim_course", "dim_satisfaction_type",
              "fact_returns", "fact_training", "fact_staff_satisfaction"]:
        n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t}", con)["n"].iloc[0]
        overzicht.append({"tabel": t, "rijen": int(n)})
        log_event("INFO", "ROWCOUNT", t, "COUNT", int(n), "Rijtelling DWH")
    con.close()

    log_event("INFO", "RUN", "", "END", "", "Extra DWH ETL klaar")
    print("\nPijplijn klaar.")
    return pd.DataFrame(overzicht)

run_extra_pipeline()

Basis-DWH is aanwezig. We kunnen verder.
Nieuwe tabellen staan klaar.
dim_return_reason: 5 nieuw, 0 gewijzigd (SCD2).
dim_course: 9 nieuw, 0 gewijzigd (SCD2).


dim_date: 81 ontbrekende retourdatums toegevoegd.


fact_returns: 690 rijen geladen.
fact_training: 402 rijen geladen (29 zonder verkooprol).
fact_staff_satisfaction: 301 rijen geladen.
Analyse-views aangemaakt.


CSV-bestanden geschreven naar: C:\Users\skyde\Documents\Github\DEAI-SE4\Great_Outdoors\powerbi_exports

Pijplijn klaar.


,tabel,rijen
0,dim_return_reason,5
1,dim_course,9
2,dim_satisfaction_type,5
3,fact_returns,690
4,fact_training,402
5,fact_staff_satisfaction,301


## 17. Voorbeeldanalyses (controle van de resultaten)

In [17]:
def q(sql):
    con = connect(DWH_DB)
    df = pd.read_sql_query(sql, con)
    con.close()
    return df

# Top 5 retourredenen op waarde
q("SELECT * FROM vw_retouren_per_reden ORDER BY retour_waarde DESC")

,reason_description,reason_category,retour_aantal,retour_waarde,aantal_retouren
0,Unsatisfactory product,Kwaliteit,5642,495650.90,120
1,Wrong product ordered,Klantkeuze,4124,314733.52,68
2,Wrong product shipped,Logistiek,2432,211829.60,62
3,Defective product,Kwaliteit,1008,94510.82,238
4,Incomplete product,Logistiek,758,63134.32,202


In [18]:
# Top 10 producten met het hoogste retourpercentage (minimaal redelijk verkocht)
q('''
SELECT product_name, product_line, verkocht_aantal, retour_aantal, retour_percentage
FROM vw_retourpercentage_per_product
WHERE verkocht_aantal >= 100
ORDER BY retour_percentage DESC
LIMIT 10
''')

,product_name,product_line,verkocht_aantal,retour_aantal,retour_percentage
0,EverGlow Lamp,Camping Equipment,40696,1000,2.46
1,Firefly Lite,Camping Equipment,13558,312,2.30
2,Granite Ice,Mountaineering Equipment,8872,194,2.19
3,TrailChef Deluxe Cook Set,Camping Equipment,3744,78,2.08
4,TrailChef Utensils,Camping Equipment,15928,328,2.06
5,Star Gazer 6,Camping Equipment,10504,214,2.04
6,Firefly Rechargeable Battery,Mountaineering Equipment,22840,430,1.88
7,Hibernator Self - Inflating Mat,Camping Equipment,11482,214,1.86
8,Polar Wave,Personal Accessories,7778,144,1.85
9,Star Peg,Camping Equipment,44686,792,1.77


In [19]:
# Cursusdeelname en (aangenomen) kosten per cursus
q("SELECT * FROM vw_cursus_deelname ORDER BY aantal_deelnames DESC")

,course_description,course_category,standard_cost,aantal_deelnames,aantal_medewerkers,totale_kosten
0,GO Communication,Communicatie,400.0,78,78,31200.0
1,GO Marketing 1,Marketing,750.0,66,66,49500.0
2,GO Marketing 2,Marketing,750.0,55,55,41250.0
3,GO Marketing 3,Marketing,750.0,53,53,39750.0
4,GO Sales 1,Sales,600.0,45,45,27000.0
5,GO Orientation,Orientatie,250.0,33,33,8250.0
6,GO Management 1,Management,1000.0,30,30,30000.0
7,GO Sales 2,Sales,600.0,23,23,13800.0
8,GO Management 2,Management,1000.0,19,19,19000.0


In [20]:
# Gemiddelde medewerkertevredenheid per jaar
q("SELECT * FROM vw_tevredenheid_per_jaar ORDER BY year_num")

,year_num,gem_tevredenheid,aantal_metingen
0,2004,3.57,101
1,2005,3.43,101
2,2006,3.65,99


In [21]:
# Hangt meer cursussen volgen samen met hogere tevredenheid?
q('''
SELECT
    CASE WHEN aantal_cursussen = 0 THEN '0 cursussen'
         WHEN aantal_cursussen BETWEEN 1 AND 3 THEN '1-3 cursussen'
         ELSE '4+ cursussen' END AS groep,
    COUNT(*)                  AS aantal_medewerkers,
    ROUND(AVG(gem_tevredenheid), 2) AS gem_tevredenheid
FROM vw_training_vs_tevredenheid
GROUP BY groep
ORDER BY groep
''')

,groep,aantal_medewerkers,gem_tevredenheid
0,0 cursussen,5,2.93
1,1-3 cursussen,28,3.48
2,4+ cursussen,68,3.60


## 18. Uitleg voor het assessment

**Inlaadstrategie**
* Feiten (`fact_returns`, `fact_training`, `fact_staff_satisfaction`): **full reload** — elke run
  worden ze geleegd en opnieuw gevuld. Eenvoudig en altijd consistent.
* `dim_satisfaction_type`: **SCD-type 1** (overschrijven, geen historie).
* `dim_return_reason` en `dim_course`: **SCD-type 2** (historie via `valid_from`, `valid_to`,
  `is_current`, `version`).

**Afgeleide dimensie-attributen (≥ 3)**
1. `dim_return_reason.reason_category` (Kwaliteit / Logistiek / Klantkeuze)
2. `dim_course.course_category` (Orientatie / Communicatie / Sales / Marketing / Management)
3. `dim_course.standard_cost` (aangenomen kost per cursus)
4. `dim_satisfaction_type.satisfaction_score` (numerieke score 1–5)

**Afgeleide meetwaarden (≥ 3)**
1. `fact_returns.return_value` = aantal × verkoopprijs (gederfde omzet)
2. `fact_returns.gross_loss` = aantal × (verkoopprijs − kostprijs) (gederfde marge)
3. `fact_returns.original_quantity` (besteld aantal, basis voor het retourpercentage)
4. `fact_training.course_cost` (kost van de gevolgde cursus)

**Database-overschrijdende associaties (de casus-uitdaging)**
* `staff.training.SALES_REPRESENTATIVE_CODE` ⇆ `sales.sales_staff.SALES_STAFF_CODE` — andere
  tabel- en kolomnaam, maar dezelfde persoon/code (zie `dim_sales_staff`).
* `sales.returned_item.ORDER_DETAIL_CODE` ⇆ `sales.order_details.ORDER_DETAIL_CODE` (via `fact_order_sales`).

**Testprocedure (nieuwe data ná 1 oktober 2025)**
* Nieuwe retouren/trainingen → bij de volgende run automatisch meegeladen (full reload feiten).
* Gewijzigde cursus- of reden-omschrijving → SCD-type 2 maakt een **nieuwe versie** aan
  (oude rij blijft met `is_current = 0` staan).
* Verwijderde brongegevens → verdwijnen bij de full reload uit de feiten.

**Belangrijke kanttekeningen (eerlijk benoemd)**
* De bron bevat **geen cursuskosten**; `standard_cost` is een **aanname** per categorie.
* Trainings-/tevredenheidsdata loopt **2004–2006**, terwijl de verkoopdata **2023–2025** is.
  Een direct verband "training → verkoop in hetzelfde jaar" is dus niet mogelijk; het verband
  training ⇆ tevredenheid (beide 2004–2006) is wél zuiver te analyseren.